# Resonator Power-Sweep Analysis — Internal Loss $1/Q_i$ vs Device Power

This notebook takes raw VNA magnitude/phase sweeps of two superconducting
microwave resonators and turns them into a power-dependent internal-loss curve,
$1/Q_i(P)$, fit to a two-level-system (TLS) saturation model.

Every scan (one frequency sweep at one power) is fit with a notch/shunt
resonator model — an algebraic circle fit for the starting guess, then a full
complex nonlinear least-squares polish. Each scan's uncertainty combines two
independent pieces, and **every $Q_i$ error bar in this notebook shows the two
added in quadrature**:

* **fit covariance** — how precisely that one scan is fit from its own data
  (Probst *et al.*, *Rev. Sci. Instrum.* **86**, 024706 (2015), Sec. IV);
* **TLS temporal fluctuation** — how much $Q_i$ itself drifts between repeated
  measurements at fixed power (Chen *et al.*, *J. Appl. Phys.* **137**, 044401
  (2025)), measured directly from dedicated time sweeps and carried to every
  scan's power by a fitted scaling law.

**Data availability & how to reproduce.** The raw data ship alongside this
notebook: one folder per sweep inside a single data root (`pwr_sweep_*` for the
power sweeps, `time_sweep_*` for the repeated fixed-power sweeps), each holding
the VNA CSV exports exactly as measured. Point `CANDIDATES` in Step 1 at the
data root and *Run All*: the notebook is a straight top-to-bottom pipeline (raw
CSVs → per-scan fits → filtered set → error model → TLS fit → figures), needs
only `numpy`, `scipy` and `matplotlib`, and saves every figure it shows as a
PNG next to the notebook. Device power always means the VNA set power plus the
recorded manual attenuation plus the fixed fridge attenuation ($-45$ dB).

**Measurement tiers.** Scans are grouped by their instrument settings — the
manual attenuator in the input line and the VNA IF bandwidth — with tier keys
that state those settings (`m30_bw100`, `m30_bw10`, `m70`) and legends that
spell them out (e.g. "$-70$ dB atten, 5–100 Hz IFBW"). Step 1 prints the full
per-sweep settings table: attenuation, span, IFBW and trace averages.

**Part I — From raw data to quality factors**

| Step | What happens |
|---|---|
| 1 | Imports and measurement configuration |
| 2 | Load raw magnitude/phase into complex $S_{21}$ |
| 3 | Dip finding, linewidth, off-resonant background |
| 4 | Cable-delay calibration per attenuation path |
| 5 | Circle fit and full complex least squares |
| 6 | Fit every scan |
| 7 | Quality filters |

**Part II — Error model**

| Step | What happens |
|---|---|
| 8 | Per-scan fit covariance |
| 9 | TLS temporal fluctuation from the time sweeps, with a fitted power-scaling law |

**Part III — Results and diagnostics**

| Step | What happens |
|---|---|
| 10 | Weighted TLS fit — free $\beta$, and $\beta$ fixed at the standard-tunneling-model value |
| 11 | Publication plots: $1/Q_i$ vs device power and vs photon number |
| 12 | SNR convergence diagnostics |
| 13 | Summary table |
| 14 | Individual fits: magnitude, phase and IQ, one plot per scan |

**Physics model** (notch-type / shunt resonator):

$$S_{21}(f) = a\,e^{i\alpha}\,e^{-2\pi i (f-\bar f)\tau}\left[1 - \frac{(Q_L/|Q_c|)e^{i\phi}}{1 + 2iQ_L(f/f_r - 1)}\right],
\qquad \frac{1}{Q_i} = \frac{1}{Q_L} - \frac{\cos\phi}{|Q_c|}.$$


## Step 1 — Imports and measurement configuration

Each `pwr_sweep_*` folder holds many power points for one resonator at one
manual-attenuation setting; the `time_sweep_*` folders used in Step 9 hold
repeated scans at fixed power. All are expected under `DATA_ROOT`.

Device power is the VNA's set power plus the total attenuation on the input
line — the manual attenuator recorded per sweep, plus a fixed attenuation
built into the fridge wiring (`FRIDGE_ATTEN`). Spans and IF bandwidths are
recorded per sweep for provenance; the frequency axis itself is reconstructed
as `linspace(fc - span/2, fc + span/2, N)`, since the raw CSVs carry no
frequency column of their own.

The cell ends by printing the **per-sweep measurement settings table**
(attenuator, span, IF bandwidth, trace averages) that accompanies the data
release; the tier labels used in every figure are generated from these
settings. The `avg` column is a placeholder (`TODO`) until the trace-average
counts are filled in from the lab notes.


In [ ]:
import os, re, glob, time
import numpy as np
from scipy.optimize import least_squares, minimize
import matplotlib.pyplot as plt

# Okabe-Ito colour-blind-safe palette, used by every figure in the notebook.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]

# One consistent figure style for the whole notebook (every plot inherits this).
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=OKABE_ITO),
    "figure.dpi": 140, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9.5, "axes.titlesize": 10.5, "axes.labelsize": 10,
    "legend.fontsize": 8.5, "legend.frameon": False,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
})

CANDIDATES = [
    "/home/pegan/Downloads/power depem q (new)",
    "data/power depem q (new)",
    "power depem q (new)",
]
DATA_ROOT = CANDIDATES[0]
for candidate_path in CANDIDATES:
    if os.path.isdir(candidate_path):
        DATA_ROOT = candidate_path
        break
print("DATA_ROOT =", DATA_ROOT, "  exists:", os.path.isdir(DATA_ROOT))

CONFIG = {
    "res1": dict(fc=6.901285e9, sweeps=[
        dict(name="pwr_sweep_045 (res 1)",    tier="m30_bw100", atten=-30.0, span=0.03e6, bw=100.0, avg=10),
        dict(name="pwr_sweep_021 (res1 low)", tier="m30_bw10", atten=-30.0, span=0.10e6, bw=10.0, avg=100),
        dict(name="pwr_sweep_047",            tier="m70",      atten=-70.0, span=0.03e6, bw=100.0, avg=20),
        dict(name="pwr_sweep_048",            tier="m70",      atten=-70.0, span=0.03e6, bw=10.0, avg=20),
        dict(name="pwr_sweep_049",            tier="m70",      atten=-70.0, span=0.03e6, bw=5.0, avg=20)]),
    "res2": dict(fc=6.718e9, sweeps=[
        dict(name="pwr_sweep_014 (res2)",     tier="m30_bw100", atten=-30.0, span=0.10e6, bw=100.0, avg=10),
        dict(name="pwr_sweep_029 (res2 low)", tier="m30_bw10", atten=-30.0, span=0.10e6, bw=10.0, avg=100),
        dict(name="pwr_sweep_044",            tier="m70",      atten=-70.0, span=0.03e6, bw=100.0, avg=20),
        dict(name="pwr_sweep_045",            tier="m70",      atten=-70.0, span=0.03e6, bw=10.0, avg=20),
        dict(name="pwr_sweep_046",            tier="m70",      atten=-70.0, span=0.03e6, bw=5.0, avg=20)]),
}

# Fixed fridge attenuation (input-line attenuators built into the fridge wiring,
# separate from the manual/switchable attenuators recorded per sweep above).
# Added to every scan's device-power calculation.
FRIDGE_ATTEN = -45.0

# Human-readable tier labels, generated from the recorded settings so every
# legend and table describes the measurement itself rather than an internal key.
def build_tier_labels(config):
    tier_settings = {}
    for res in config:
        for sweep in config[res]["sweeps"]:
            attens, bws = tier_settings.setdefault(sweep["tier"], (set(), set()))
            attens.add(sweep["atten"])
            bws.add(sweep["bw"])
    labels = {}
    for tier, (attens, bws) in tier_settings.items():
        atten_txt = "/".join(f"{a:.0f} dB" for a in sorted(attens))
        bw_sorted = sorted(bws)
        if len(bw_sorted) == 1:
            bw_txt = f"{bw_sorted[0]:.0f} Hz"
        else:
            bw_txt = f"{bw_sorted[0]:.0f}\u2013{bw_sorted[-1]:.0f} Hz"
        labels[tier] = f"{atten_txt} atten, {bw_txt} IFBW"
    return labels

TIER_LABEL = build_tier_labels(CONFIG)
RES_TITLE = {"res1": "Resonator 1", "res2": "Resonator 2"}

# Per-sweep measurement settings, printed for the data release.
# avg = VNA trace averages, as recorded in the lab notes.
print(f"{'resonator':>9s}  {'sweep folder':28s} {'tier':10s} {'atten':>6s} {'span':>9s} {'IFBW':>8s} {'avg':>5s}")
for res in CONFIG:
    for sweep in CONFIG[res]["sweeps"]:
        avg_txt = "TODO" if sweep["avg"] is None else str(sweep["avg"])
        print(f"{res:>9s}  {sweep['name']:28s} {sweep['tier']:10s} {sweep['atten']:+5.0f}  "
              f"{sweep['span']/1e3:6.0f} kHz {sweep['bw']:5.0f} Hz {avg_txt:>5s}")
print()

LINEWIDTH_MAX_SPANS = 3.0
QL_MEDIAN_JUMP      = 10.0
FNAME_RX = re.compile(r"mag_p([+-][\d.]+)dB_req_p([+-][\d.]+)dB_set\.csv$")

## Step 2 — Load raw magnitude/phase into complex $S_{21}$

Magnitude is stored in dB, so amplitude is $|S_{21}|=10^{\rm dB/20}$; phase is
in radians, unless a file's values exceed $2\pi$, in which case it's assumed
to be in degrees and converted.


In [ ]:
def load_sweep(swdir):
    scans = []
    for fmag in glob.glob(os.path.join(swdir, "mag_p*dB_set.csv")):
        m = FNAME_RX.search(os.path.basename(fmag))
        if not m:
            continue
        p_req, p_set = float(m.group(1)), float(m.group(2))
        mag_db = np.loadtxt(fmag)
        ph = np.loadtxt(fmag.replace("mag_", "phase_"))
        if np.nanmax(np.abs(ph)) > 2.0 * np.pi + 0.5:
            ph = np.deg2rad(ph)
        z = 10.0 ** (mag_db / 20.0) * np.exp(1j * ph)
        scans.append(dict(p_req=p_req, p_set=p_set, z=z, n=len(z), mag_db=mag_db, phase=ph))

    def get_set_power(scan):
        return scan["p_set"]

    scans.sort(key=get_set_power)
    return scans

def load_resonator(res):
    cfg, sweeps = CONFIG[res], []
    for sw in cfg["sweeps"]:
        scans = load_sweep(os.path.join(DATA_ROOT, sw["name"]))
        for scan in scans:
            scan["f"] = np.linspace(cfg["fc"] - sw["span"] / 2, cfg["fc"] + sw["span"] / 2, scan["n"])
        sweeps.append(dict(sw, scans=scans))
        print(f"  {sw['name']:26s} tier={sw['tier']:7s} n={len(scans):3d} "
              f"span={sw['span']/1e3:.0f} kHz bw={sw['bw']:.0f} Hz atten={sw['atten']:+.0f} dB")
    return sweeps

DATA = {}
for res in CONFIG:
    print(f"[{res}]  f_c = {CONFIG[res]['fc']/1e9:.6f} GHz")
    DATA[res] = load_resonator(res)
    print()

## Step 3 — Dip finding, linewidth, off-resonant background

A boxcar-smoothed $|S_{21}|^2$ locates the resonance dip, and a parabolic fit
around the minimum refines its position to sub-bin precision. The 3 dB
half-power crossings either side of the dip give a rough linewidth; if only
one side of the dip is available in the data, that side's crossing is mirrored
to estimate the other.

This step also identifies which points sit far enough from the dip to count as
"off-resonant" background — used both to normalize each scan before fitting,
and as a fallback if a scan is missing one wing of usable baseline: the search
can be restricted to only the left or only the right side of the dip,
whichever the data actually supports.


In [ ]:
def rough_char(f, z, span, side=None):
    n = len(f)
    p = np.abs(z) ** 2
    k = max(3, n // 100) | 1
    ps = np.convolve(p, np.ones(k) / k, mode="same")
    i0 = int(np.argmin(ps[k:n - k])) + k
    if 0 < i0 < n - 1:
        y1, y2, y3 = np.log(ps[i0 - 1:i0 + 2] + 1e-30)
        den = y1 - 2 * y2 + y3
        di = np.clip(0.5 * (y1 - y3) / den, -1, 1) if abs(den) > 1e-12 else 0.0
    else:
        di = 0.0
    fr0 = f[i0] + di * (f[1] - f[0])
    m = max(5, n // 10)
    base = max(np.median(np.concatenate([ps[:m], ps[-m:]])), ps[i0] * 1.0000001)
    half = 0.5 * (base + ps[i0])
    above = ps > half
    li, ri = np.flatnonzero(above[:i0]), np.flatnonzero(above[i0:])
    hl = f[i0] - f[li[-1]] if li.size else np.nan
    hr = f[ri[0] + i0] - f[i0] if ri.size else np.nan
    if np.isnan(hl) and np.isnan(hr):
        fw = span / 4.0
    elif np.isnan(hl):
        fw = 2 * hr
    elif np.isnan(hr):
        fw = 2 * hl
    else:
        fw = hl + hr
    fw = float(np.clip(fw, 2 * (f[1] - f[0]), 4 * span))
    off = np.abs(f - fr0) > max(2.5 * fw, 0.12 * span)

    side_mask = np.ones(n, dtype=bool)
    if side == "right":
        side_mask = f > fr0
    elif side == "left":
        side_mask = f < fr0
    off = off & side_mask

    if off.sum() < 12:
        cand = np.flatnonzero(side_mask)
        if len(cand) >= 12:
            dist = np.abs(f[cand] - fr0)
            thresh = np.quantile(dist, 0.5)
            off = np.zeros(n, dtype=bool)
            off[cand[dist >= thresh]] = True
    return fr0, fw, off

## Step 4 — Cable delay, calibrated on the highest-SNR data

Cable delay $\tau$ shows up as a frequency-dependent phase rotation that needs
to be removed before a circle fit makes sense. A simple phase-slope fit
against off-resonant points doesn't work well here — the sweep spans are
narrow enough that the resonance itself contaminates the slope estimate.

$\tau$ is instead fit as a free parameter of the full notch model, using the
highest-SNR scans on each attenuation path. For most paths, the 10
highest-power scans are each fit independently and the median is locked in as
that path's delay. The two $-30$ dB tiers share one delay (they use the same
attenuation path); the very-low tier gets its own — except for res1's very-low
path, where the single best-SNR scan's own fit is used directly rather than a
median: individual delay estimates on that narrow a span scatter over a wide
range from scan to scan, so averaging several of them together isn't obviously
better than trusting the one scan with the most information in it.


In [ ]:
def taubin_circle(x, y):
    xm, ym = x.mean(), y.mean()
    u, v = x - xm, y - ym
    zsq = u * u + v * v
    Zm = zsq.mean()
    Z0 = (zsq - Zm) / (2.0 * np.sqrt(Zm) + 1e-30)
    _, _, Vt = np.linalg.svd(np.vstack([Z0, u, v]).T, full_matrices=False)
    a0, b1, b2 = Vt[-1]
    a0 /= (2.0 * np.sqrt(Zm) + 1e-30)
    xc = -b1 / (2 * a0 + 1e-30) + xm
    yc = -b2 / (2 * a0 + 1e-30) + ym
    return xc, yc, float(np.sqrt(np.mean((x - xc) ** 2 + (y - yc) ** 2)))

def notch(f, fr, ql, qc, phi, ar, ai):
    x = 2.0 * ql * (f / fr - 1.0)
    return (ar + 1j * ai) * (1.0 - (ql / qc) * np.exp(1j * phi) / (1.0 + 1j * x))

def seed(f, zn, span, side=None):
    fr0, fw, off = rough_char(f, zn, span, side=side)
    xc, yc, r0 = taubin_circle(zn.real, zn.imag)
    d = float(np.clip(2 * r0, 1e-6, 50.0))
    phi0 = float(np.angle(1 - (xc + 1j * yc)))
    ql0 = float(np.clip(fr0 / fw, 10.0, 1e9))
    return fr0, ql0, float(np.clip(ql0 / d, 10.0, 1e12)), phi0

def calib_delay_scan(f, z, span, tau_max=1e-6, side=None):
    fr0, fw, off = rough_char(f, z, span, side=side)
    b0 = np.median(z[off].real) + 1j * np.median(z[off].imag)
    if abs(b0) < 1e-12:
        return None
    zn = z / b0
    fr0, ql0, qc0, phi0 = seed(f, zn, span, side=side)
    fm = f.mean()
    p0 = [fr0, ql0, qc0, phi0, 1.0, 0.0, 0.0]
    lo = [f.min() - 2 * span, 1.0, 1.0, -np.pi, -10.0, -10.0, -tau_max]
    hi = [f.max() + 2 * span, 1e9, 1e13,  np.pi,  10.0,  10.0,  tau_max]
    def resid(p):
        m = np.exp(-2j * np.pi * (f - fm) * p[6]) * notch(f, *p[:6]) - zn
        return np.concatenate([m.real, m.imag])
    try:
        sol = least_squares(resid, np.clip(p0, lo, hi), bounds=(lo, hi),
                            x_scale=[span, ql0, qc0, 1, 1, 1, 1e-9],
                            ftol=1e-13, xtol=1e-13, max_nfev=800)
    except Exception:
        return None
    tau = sol.x[6]
    return None if abs(tau) > 0.99 * tau_max else float(tau)

def path_delay(scans, k_top=10, side=None):
    '''Fit delay independently on the k_top highest-power scans of a path,
    then take the median as the locked-in value for that path.'''
    def get_power(scan_tuple):
        f, z, span, p_set = scan_tuple
        return p_set

    scans_by_power = sorted(scans, key=get_power, reverse=True)
    best_scans = scans_by_power[:k_top]

    taus = []
    for f, z, span, p_set in best_scans:
        tau = calib_delay_scan(f, z, span, side=side)
        if tau is not None:
            taus.append(tau)

    if not taus:
        return 0.0, 0
    return float(np.median(taus)), len(taus)

def path_delay_best_snr(scans, side=None):
    '''Use the single scan with the best data-quality SNR directly, rather
    than a median over several scans -- for cases where a median across
    scans of wildly varying quality isn't a well-defined thing to average.'''
    def snr_of_scan(scan_tuple):
        f, z, span, p_set = scan_tuple
        xc, yc, r0 = taubin_circle(z.real, z.imag)
        radial_distance = np.abs(z - (xc + 1j * yc))
        radial_scatter = np.sqrt(np.sum((radial_distance - r0) ** 2) / (len(radial_distance) - 1))
        return r0 / (radial_scatter + 1e-300)

    best_scan = max(scans, key=snr_of_scan)
    f, z, span, p_set = best_scan
    tau = calib_delay_scan(f, z, span, side=side)
    if tau is None:
        return 0.0, 0
    return float(tau), 1

## Step 5 — Circle fit in the IQ plane, then full complex least squares

An algebraic (Taubin) circle fit gives a closed-form starting guess — the
circle's diameter sets $Q_L/|Q_c|$, and its centre offset sets $\phi$. That
seed is then refined by a full nonlinear least-squares fit over all six model
parameters ($f_r$, $Q_L$, $|Q_c|$, $\phi$, and a complex background).

$$Q_L \to |Q_c| \to \frac{1}{Q_i} = \frac{1}{Q_L} - \frac{\cos\phi}{|Q_c|}$$

The error on $1/Q_i$ propagates the *full* covariance matrix from the fit, not
just the diagonal variances, since $Q_L$, $Q_c$ and $\phi$ are correlated with
each other.

SNR is reported following Probst Eq. 14: the fitted circle's radius divided by
the RMS radial scatter of the data around it.

For one specific tier — res1's lowest-power scans — the data doesn't reach a
stable baseline on one side of the resonance before the sweep ends, so the
background/normalization step there only uses points from the side that does
reach baseline (`BG_SIDE`).


In [ ]:
BG_SIDE = {"pwr_sweep_047": "right", "pwr_sweep_048": "right", "pwr_sweep_049": "right"}

def core_fit(f, zn, span, p0=None, side=None):
    if p0 is None:
        fr0, ql0, qc0, phi0 = seed(f, zn, span, side=side)
        p0 = [fr0, ql0, qc0, phi0, 1.0, 0.0]
    else:
        fr0, ql0, qc0 = p0[0], p0[1], p0[2]
    lo = [f.min() - 2 * span, 1.0, 1.0, -np.pi, -10.0, -10.0]
    hi = [f.max() + 2 * span, 1e9, 1e13,  np.pi,  10.0,  10.0]
    def resid(p):
        m = notch(f, *p) - zn
        return np.concatenate([m.real, m.imag])
    try:
        return least_squares(resid, np.clip(p0, lo, hi), bounds=(lo, hi), method="trf",
                             x_scale=[span, ql0, qc0, 1, 1, 1],
                             ftol=1e-12, xtol=1e-12, max_nfev=400)
    except Exception:
        return None

def qi_and_error(sol):
    '''Probst Sec. IV: cov = chi2/(N-lambda) * (J^T J)^-1, then propagate to 1/Q_i.'''
    fr, ql, qc, phi, ar, ai = sol.x
    J = sol.jac
    dof = max(J.shape[0] - J.shape[1], 1)
    s2 = 2.0 * sol.cost / dof
    try:
        cov = np.linalg.inv(J.T @ J) * s2
    except np.linalg.LinAlgError:
        return None
    inv_qi = 1.0 / ql - np.cos(phi) / qc
    if not np.isfinite(inv_qi) or inv_qi == 0:
        return None
    qi = 1.0 / inv_qi
    g = np.zeros(6)
    g[1] = -1.0 / ql ** 2
    g[2] =  np.cos(phi) / qc ** 2
    g[3] =  np.sin(phi) / qc
    var_inv = float(g @ cov @ g)
    sig_ql = float(np.sqrt(max(cov[1, 1], 0.0)))
    sig_qc = float(np.sqrt(max(cov[2, 2], 0.0)))
    return qi, np.sqrt(max(var_inv, 0.0)) * qi ** 2, sig_ql, sig_qc, s2

def snr_eq14(f, zn, p):
    fr, ql, qc, phi, ar, ai = p
    b = ar + 1j * ai
    c = b * (1 - 0.5 * (ql / qc) * np.exp(1j * phi))
    r0 = abs(b) * 0.5 * ql / qc
    ri = np.abs(zn - c)
    sig_r = np.sqrt(np.sum((ri - r0) ** 2) / (len(ri) - 1))
    return float(r0 / (sig_r + 1e-300))

def fit_scan(f, z, tau, span, sweep_name=None):
    z = z * np.exp(2j * np.pi * (f - f.mean()) * tau)
    side = BG_SIDE.get(sweep_name)
    fr0, fw, off = rough_char(f, z, span, side=side)
    b0 = np.median(z[off].real) + 1j * np.median(z[off].imag)
    if abs(b0) < 1e-12:
        return None
    zn = z / b0
    sol = core_fit(f, zn, span, side=side)
    if sol is None:
        return None
    r = qi_and_error(sol)
    if r is None:
        return None
    qi, sig_qi, sig_ql, sig_qc, s2 = r
    fr, ql, qc, phi, ar, ai = sol.x
    snr = snr_eq14(f, zn, sol.x)
    model = notch(f, *sol.x)

    return dict(fr=fr, ql=ql, qc=qc, phi=phi, qi=qi, sig_qi=sig_qi,
                sig_ql=sig_ql, sig_qc=sig_qc, inv_qi=1.0 / qi, snr=snr,
                chi2red=s2, zn=zn, zn_model=model)

## Step 6 — Fit every scan

Delay is locked per attenuation path (Step 4), then every scan is fit
independently. Around 250 scans total, a few seconds of work.


In [ ]:
t_start = time.time()
FITS, TAUS = {}, {}
for res in CONFIG:
    sweeps = DATA[res]

    high_tier_scans = []
    verylow_tier_scans = []
    for sweep in sweeps:
        for scan in sweep["scans"]:
            scan_tuple = (scan["f"], scan["z"], sweep["span"], scan["p_set"])
            if sweep["tier"] == "m30_bw100":
                high_tier_scans.append(scan_tuple)
            elif sweep["tier"] == "m70":
                verylow_tier_scans.append(scan_tuple)

    verylow_needs_right_side_only = any(
        sweep["name"] in BG_SIDE for sweep in sweeps if sweep["tier"] == "m70"
    )
    verylow_side = "right" if verylow_needs_right_side_only else None

    tau30, n30 = path_delay(high_tier_scans)
    if res == "res1":
        tau70, n70 = path_delay_best_snr(verylow_tier_scans, side=verylow_side)
    else:
        tau70, n70 = path_delay(verylow_tier_scans, side=verylow_side)
    TAUS[res] = (tau30, tau70)

    print(f"[{res}] tau(-30 dB path) = {tau30*1e9:+8.2f} ns   from {n30} high-tier scans -> shared with low")
    delay_source_label = "single highest-SNR scan" if res == "res1" else f"{n70} verylow scans (median)"
    print(f"[{res}] tau(-70 dB path) = {tau70*1e9:+8.2f} ns   from {delay_source_label}  -> dedicated")

    fit_results = []
    for sweep in sweeps:
        tau = tau70 if sweep["tier"] == "m70" else tau30
        for scan in sweep["scans"]:
            result = fit_scan(scan["f"], scan["z"], tau, sweep["span"], sweep_name=sweep["name"])
            if result is None:
                continue
            device_power = scan["p_set"] + sweep["atten"] + FRIDGE_ATTEN
            result.update(sweep=sweep["name"], tier=sweep["tier"], atten=sweep["atten"], span=sweep["span"], bw=sweep["bw"],
                          p_dev=device_power, f=scan["f"])
            fit_results.append(result)
    FITS[res] = fit_results
    print(f"[{res}] fitted {len(fit_results)} scans\n")
print(f"fitting time: {time.time() - t_start:.1f} s")

## Step 7 — Quality filters

A handful of checks catch fits that converged to something unphysical or
degenerate, without touching anything that's just noisy:

1. **Resolvable linewidth**: $f_r/Q_L \le 3\times$ the sweep span. There's no
   lower bound — a narrow linewidth relative to span is a *good* sign (a
   well-resolved, high-$Q$ resonance), not a defect.
2. **Mathematical validity**: $Q_i$, $Q_L$, $Q_c$ all finite and positive.
3. **Degenerate collapse**: $Q_L$ within 10× of its own tier's median
   (computed from whatever survives checks 1–2), so a real difference between
   measurement conditions is never mistaken for a bad fit.
4. **Sweep-local outlier**: $1/Q_i$ within 10× of the median $1/Q_i$ for its
   own sweep file. This catches a specific low-SNR failure mode where a fit
   converges cleanly, with a small formal error, to the wrong answer.


In [ ]:
for res in CONFIG:
    all_fits = FITS[res]

    for fit in all_fits:
        linewidth_ok = (fit["fr"] / fit["ql"]) <= LINEWIDTH_MAX_SPANS * fit["span"]
        values_ok = np.isfinite(fit["qi"]) and fit["qi"] > 0 and fit["ql"] > 0 and fit["qc"] > 0
        fit["keep"] = bool(linewidth_ok and values_ok)
        if fit["keep"]:
            fit["why"] = ""
        elif not linewidth_ok:
            fit["why"] = "linewidth"
        else:
            fit["why"] = "Qi<=0"

    for tier in ("m30_bw100", "m30_bw10", "m70"):
        fits_in_tier = [fit for fit in all_fits if fit["tier"] == tier and fit["keep"]]
        if not fits_in_tier:
            continue
        median_ql = np.median([fit["ql"] for fit in fits_in_tier])
        for fit in fits_in_tier:
            if not (median_ql / QL_MEDIAN_JUMP <= fit["ql"] <= median_ql * QL_MEDIAN_JUMP):
                fit["keep"], fit["why"] = False, "QL collapse"

    sweep_names = sorted(set(fit["sweep"] for fit in all_fits))
    for sweep_name in sweep_names:
        fits_in_sweep = [fit for fit in all_fits if fit["sweep"] == sweep_name and fit["keep"]]
        if len(fits_in_sweep) < 4:
            continue
        median_inv_qi = np.median([fit["inv_qi"] for fit in fits_in_sweep])
        for fit in fits_in_sweep:
            if not (median_inv_qi / QL_MEDIAN_JUMP <= fit["inv_qi"] <= median_inv_qi * QL_MEDIAN_JUMP):
                fit["keep"], fit["why"] = False, "1/Qi outlier"

    kept = [fit for fit in all_fits if fit["keep"]]
    dropped = [fit for fit in all_fits if not fit["keep"]]
    if dropped:
        dropped_description = ", ".join(f"{fit['sweep'][-3:]}@{fit['p_dev']:.1f}dBm({fit['why']})" for fit in dropped)
    else:
        dropped_description = "none"
    print(f"[{res}] kept {len(kept)}/{len(all_fits)}   dropped: {dropped_description}")

## Step 8 — Error model I: per-scan fit covariance

$$\sigma_{\rm formal} = \frac{\sigma_{Q_i}}{Q_i\ln 10}\quad\text{(converted to log}_{10}\text{ space)}$$

The first error component is the formal fit covariance (Probst Sec. IV) — the
honest statistical precision of each individual fit. It reflects how
well-determined the fit is from its own data, not how much a repeated
measurement at the same power would drift or disagree. That second, physical
component is measured and added in Step 9; until then, `sig_total` is
initialised to the covariance term alone.


In [ ]:
FINAL = {}
for res in CONFIG:
    kept = [r for r in FITS[res] if r["keep"]]
    for r in kept:
        r["sig_formal"] = r["sig_qi"] / (r["qi"] * np.log(10.0))
        r["sig_total"] = r["sig_formal"]
    FINAL[res] = kept
    print(f"[{res}] n = {len(kept)}   "
          f"median rel. error = {100*np.log(10)*np.median([r['sig_total'] for r in kept]):.1f}%")
    for swname in sorted({r["sweep"] for r in kept}):
        g = [r for r in kept if r["sweep"] == swname]
        print(f"   {swname:26s} n={len(g):3d}  SNR_med={np.median([r['snr'] for r in g]):7.1f}  "
              f"rel.err_med={100*np.log(10)*np.median([r['sig_total'] for r in g]):6.1f}%")
    print()

## Step 9 — Error model II: TLS temporal fluctuation from the time sweeps

The formal covariance says nothing about how much $Q_i$ itself moves between
repeated measurements. Chen *et al.* (*J. Appl. Phys.* **137**, 044401 (2025))
show that at low photon number, TLS spectral diffusion makes $Q_i$ fluctuate
by tens of percent within minutes, and that covariance errors do not include
this.

Here that fluctuation is measured directly: repeated fixed-power scans of res2
(`time_sweep_001` at +3 dBm VNA output and `time_sweep_003` at +10 dBm, both
assumed on the $-70$ dB manual-attenuation path). Each time sweep is fit
jointly — per-scan $f_r$, $Q_L$ and complex background, but a single shared
$|Q_c|$ and $\phi$, since the coupling geometry cannot change between
back-to-back scans (the same constraint idea as the PC-CM method of Chen
*et al.*). `time_sweep_002` is excluded: it was taken immediately after the
power step, while the system was visibly still settling.

`FLUCT_STAT` selects the fluctuation statistic per sweep: the **max** pairwise
$|\Delta Q_i|/\langle Q_i\rangle$ (a conservative worst-case over the measured
window) or the **mean** pairwise value ($\approx 1.13\,\sigma$ for Gaussian
scatter). Both are printed for reference. The chosen fractions at the two
measurement powers are then turned into a **fluctuation-vs-power scaling law**
by fitting a two-parameter model through the two points; `FLUCT_MODEL` selects
which:

* `"linear"` — ${\rm fluct} = a + b\,P_{\rm dev}$ (dBm), clipped at zero;
* `"step"` — a two-level step anchored on the measurement powers, kept for
  comparison.

With only two measured points, every two-parameter model passes through both
exactly — the data cannot distinguish them, and the choice only fixes how the
term **extrapolates** outside the measured window. The figure at the end of
this step shows the data behind the anchors: the per-scan wander over time
(left) and the full distribution of pairwise $|\Delta Q_i|/\langle Q_i\rangle$
at each measured power (right), with the scaling laws overlaid and their
predictions printed at the dataset extremes.

The term is added in quadrature for every scan of both resonators:

$$\sigma_{\rm total} = \sqrt{\sigma_{\rm formal}^2 +
\left(\frac{{\rm fluct}\,/\,\langle Q_i\rangle}{\ln 10}\right)^2}$$

res1 has no time sweeps of its own, so the res2-measured fluctuation is
applied to it as a proxy. The per-sweep fit results are kept in `TS_RESULTS`
and feed the $\Delta Q_i$ figure at the end of this step.


In [ ]:
# Time sweeps: repeated fixed-power scans of res2, used to measure how much Q_i
# fluctuates between back-to-back scans (TLS spectral diffusion, Chen et al. 2025).
TIME_SWEEPS = ["time_sweep_001", "time_sweep_003"]  # 002 excluded: taken while still settling after the power step
TIMESWEEP_ATTEN = -70.0  # manual attenuation assumed in place for the time-sweep measurements
FLUCT_STAT = "max"  # "max": worst-case pairwise |dQi| (conservative); "mean": typical pairwise |dQi| (~1.13 sigma)

def load_time_sweep(name):
    swdir = os.path.join(DATA_ROOT, name)
    f = np.loadtxt(os.path.join(swdir, "freqs.csv"))
    mag_db = np.atleast_2d(np.loadtxt(os.path.join(swdir, "magnitude_matrix.csv"), delimiter=","))
    ph = np.atleast_2d(np.loadtxt(os.path.join(swdir, "phase_matrix.csv"), delimiter=","))
    if np.nanmax(np.abs(ph)) > 2.0 * np.pi + 0.5:
        ph = np.deg2rad(ph)
    p_req = float(np.loadtxt(os.path.join(swdir, "power_requested.csv")))
    t_rel = np.atleast_1d(np.loadtxt(os.path.join(swdir, "timestamps.csv")))
    t_rel = t_rel - t_rel[0]
    z = 10.0 ** (mag_db / 20.0) * np.exp(1j * ph)
    return f, z, p_req, t_rel

def fit_time_sweep(f, z_scans, tau):
    """Joint fit of all scans in one time sweep: per-scan f_r, Q_L and complex
    background, but a single shared |Q_c| and phi. Coupling geometry does not
    change between back-to-back scans, and at this SNR a per-scan free fit
    leaves Q_c/phi degenerate with the scan-to-scan baseline wander -- sharing
    them pins the geometry so the fluctuation lands in Q_i where it belongs
    (same constraint idea as the PC-CM method of Chen et al. 2025)."""
    span = float(f[-1] - f[0])
    n_scans = z_scans.shape[0]
    z_corr = z_scans * np.exp(2j * np.pi * (f - f.mean()) * tau)[None, :]

    scan_seeds, scan_norms = [], []
    for i in range(n_scans):
        fr0, fw, off = rough_char(f, z_corr[i], span)
        b0 = np.median(z_corr[i][off].real) + 1j * np.median(z_corr[i][off].imag)
        scan_norms.append(b0)
        scan_seeds.append(seed(f, z_corr[i] / b0, span))

    qc0 = float(np.median([s[2] for s in scan_seeds]))
    phi0 = float(np.median([s[3] for s in scan_seeds]))
    p0, lo, hi, xsc = [qc0, phi0], [1.0, -np.pi], [1e13, np.pi], [qc0, 1.0]
    for fr0, ql0, _, _ in scan_seeds:
        p0 += [fr0, ql0, 1.0, 0.0]
        lo += [f.min() - 2 * span, 1.0, -10.0, -10.0]
        hi += [f.max() + 2 * span, 1e9, 10.0, 10.0]
        xsc += [span, ql0, 1.0, 1.0]

    def resid(p):
        qc, phi = p[0], p[1]
        parts = []
        for i in range(n_scans):
            fr, ql, ar, ai = p[2 + 4 * i: 6 + 4 * i]
            m = notch(f, fr, ql, qc, phi, ar, ai) - z_corr[i] / scan_norms[i]
            parts.append(m.real)
            parts.append(m.imag)
        return np.concatenate(parts)

    sol = least_squares(resid, np.clip(p0, lo, hi), bounds=(lo, hi), method="trf",
                        x_scale=xsc, ftol=1e-12, xtol=1e-12, max_nfev=6000)
    qc, phi = sol.x[0], sol.x[1]
    per_scan = sol.x[2:].reshape(n_scans, 4)
    fr, ql = per_scan[:, 0], per_scan[:, 1]
    qi = 1.0 / (1.0 / ql - np.cos(phi) / qc)

    # covariance in x_scale units (avoids conditioning loss across the huge
    # parameter-magnitude range), then back to physical units
    dof = max(sol.fun.size - sol.x.size, 1)
    s2 = 2.0 * sol.cost / dof
    J_scaled = sol.jac * np.array(xsc)[None, :]
    cov = s2 * np.linalg.pinv(J_scaled.T @ J_scaled) * np.outer(xsc, xsc)
    sig_qi = np.empty(n_scans)
    sig_qc = float(np.sqrt(max(cov[0, 0], 0.0)))
    for i in range(n_scans):
        g = np.zeros(len(sol.x))
        g[0] = np.cos(phi) / qc ** 2
        g[1] = np.sin(phi) / qc
        g[2 + 4 * i + 1] = -1.0 / ql[i] ** 2
        sig_qi[i] = qi[i] ** 2 * np.sqrt(max(float(g @ cov @ g), 0.0))

    # per-scan SNR with the pipeline's Eq. 14 metric, in the joint-fit frame
    snr = np.empty(n_scans)
    for i in range(n_scans):
        p6 = (fr[i], ql[i], qc, phi, per_scan[i, 2], per_scan[i, 3])
        snr[i] = snr_eq14(f, z_corr[i] / scan_norms[i], p6)

    return dict(qi=qi, ql=ql, fr=fr, qc=qc, phi=phi, sig_qi=sig_qi, sig_qc=sig_qc, snr=snr)

TLS_FLUCT_POINTS = []
print(f"fluctuation statistic in use: {FLUCT_STAT} pairwise |dQi|")
print("time sweep        P_dev      scans  <Qi>        max|dQi| (frac)     mean|dQi| (frac)")
TS_RESULTS = {}
for ts_name in TIME_SWEEPS:
    f_ts, z_ts, p_req, t_rel = load_time_sweep(ts_name)
    tau70_res2 = TAUS["res2"][1]  # time sweeps are res2 on the -70 dB path
    ts_fit = fit_time_sweep(f_ts, z_ts, tau70_res2)
    qi = ts_fit["qi"]
    pair_diffs = np.abs(qi[:, None] - qi[None, :])[np.triu_indices(len(qi), 1)]
    frac_by_stat = {"max": float(pair_diffs.max() / qi.mean()),
                    "mean": float(pair_diffs.mean() / qi.mean())}
    if FLUCT_STAT not in frac_by_stat:
        raise ValueError(f'FLUCT_STAT must be "max" or "mean" (spelled out), got {FLUCT_STAT!r}')
    fluct_frac = frac_by_stat[FLUCT_STAT]
    p_dev = p_req + TIMESWEEP_ATTEN + FRIDGE_ATTEN
    TLS_FLUCT_POINTS.append((p_dev, fluct_frac))
    TS_RESULTS[ts_name] = dict(ts_fit, t=t_rel, p_req=p_req, p_dev=p_dev)
    print(f"{ts_name:16s}  {p_dev:+7.1f} dBm  {len(qi):3d}   {qi.mean():9.0f}  "
          f"{pair_diffs.max():9.0f} ({100 * frac_by_stat['max']:5.1f}%)  "
          f"{pair_diffs.mean():9.0f} ({100 * frac_by_stat['mean']:5.1f}%)")

TLS_FLUCT_POINTS.sort()

# Two measured (device power, fluctuation) anchors -> fit a scaling law through
# them. Two points pin ANY two-parameter model exactly, so every model below
# reproduces both measurements; they only differ in how they extrapolate
# outside the measured window. The plot at the end of this cell shows that.
FLUCT_MODEL = "linear"  # "linear": a + b*P_dBm (clipped at 0) | "step": two-level step anchored on the measurement powers
(FLUCT_P1, FLUCT_F1), (FLUCT_P2, FLUCT_F2) = TLS_FLUCT_POINTS  # 001 (lower power), 003

FLUCT_SLOPE_LIN = (FLUCT_F2 - FLUCT_F1) / (FLUCT_P2 - FLUCT_P1)  # fraction per dB

def tls_fluct_frac(p_dev, model=None):
    """Relative Q_i fluctuation at a given device power, from the scaling law
    fitted through the two time-sweep measurements (selected by FLUCT_MODEL)."""
    model = FLUCT_MODEL if model is None else model
    if model == "linear":
        return float(max(FLUCT_F1 + FLUCT_SLOPE_LIN * (p_dev - FLUCT_P1), 0.0))
    if model == "step":
        return float(FLUCT_F1 if p_dev <= FLUCT_P1 else FLUCT_F2)
    raise ValueError(f'FLUCT_MODEL must be "linear" or "step", got {model!r}')

print(f"\nfluctuation scaling model in use: {FLUCT_MODEL}")
print(f"  linear:   fluct(P) = {100*FLUCT_F1:.1f}% {'+' if FLUCT_SLOPE_LIN >= 0 else '-'} "
      f"{abs(100*FLUCT_SLOPE_LIN):.2f}%/dB * (P - ({FLUCT_P1:+.1f} dBm)), clipped at 0")

for res in CONFIG:
    for r in FINAL[res]:
        r["sig_fluct"] = tls_fluct_frac(r["p_dev"]) / np.log(10.0)
        r["sig_total"] = np.sqrt(r["sig_formal"] ** 2 + r["sig_fluct"] ** 2)
    formal_med = 100 * np.log(10.0) * np.median([r["sig_formal"] for r in FINAL[res]])
    total_med = 100 * np.log(10.0) * np.median([r["sig_total"] for r in FINAL[res]])
    print(f"[{res}] median rel. error: formal {formal_med:.1f}%  ->  total {total_med:.1f}%")
print("(res2-measured fluctuation applied to both resonators; res1 has no time sweeps of its own)")

# --- delta Q_i shown with the data behind it: per-sweep wander over time (one
#     panel per time sweep, own scale each), and the pairwise |dQi| distributions
#     at the two powers with the scaling laws; the anchor marker is whichever
#     statistic FLUCT_STAT has selected ---
p_all = np.array([r["p_dev"] for res in CONFIG for r in FINAL[res]])
p_grid = np.linspace(p_all.min() - 2.0, p_all.max() + 2.0, 400)
model_style = {"linear": ("-", "#000000"), "step": (":", "0.45")}
ts_colour = {"time_sweep_001": "#E69F00", "time_sweep_003": "#0072B2"}

fig, axes = plt.subplots(1, 1 + len(TIME_SWEEPS),
                         figsize=(4.1 * len(TIME_SWEEPS) + 5.6, 4.3),
                         gridspec_kw=dict(width_ratios=[1.0] * len(TIME_SWEEPS) + [1.4]))
time_axes, ax_p = axes[:-1], axes[-1]

for ax_t, ts_name in zip(time_axes, TIME_SWEEPS):
    ts = TS_RESULTS[ts_name]
    col = ts_colour.get(ts_name, "0.4")
    dev_pct = 100.0 * (ts["qi"] / ts["qi"].mean() - 1.0)
    ax_t.plot(ts["t"], dev_pct, "o-", color=col, ms=4.5, lw=1.0)
    ax_t.axhline(0.0, color="0.6", lw=0.8)
    ax_t.set_xlabel("Time (s)")
    ax_t.set_title(f"Resonator 2 — {ts_name} ({ts['p_dev']:+.0f} dBm)", fontsize=10)
time_axes[0].set_ylabel(r"$Q_i$ deviation from sweep mean (%)")

jitter_rng = np.random.default_rng(0)
for ts_name in TIME_SWEEPS:
    ts = TS_RESULTS[ts_name]
    col = ts_colour.get(ts_name, "0.4")
    qi = ts["qi"]
    pair = np.abs(qi[:, None] - qi[None, :])[np.triu_indices(len(qi), 1)] / qi.mean()
    x_jitter = ts["p_dev"] + jitter_rng.uniform(-0.6, 0.6, pair.size)
    ax_p.plot(x_jitter, 100.0 * pair, ".", color=col, ms=5, alpha=0.6)
    anchor = pair.max() if FLUCT_STAT == "max" else pair.mean()
    ax_p.plot([ts["p_dev"]], [100.0 * anchor], "^", color=col, ms=8, mec="k", mew=0.4)
for model, (ls, col_m) in model_style.items():
    curve = np.array([tls_fluct_frac(p, model) for p in p_grid])
    ax_p.plot(p_grid, 100.0 * curve, ls, color=col_m,
              lw=2.0 if model == FLUCT_MODEL else 1.2,
              label=model + (" (in use)" if model == FLUCT_MODEL else ""))
from matplotlib.lines import Line2D
marker_handles = [Line2D([], [], color="0.3", marker=".", lw=0, ms=6,
                         label=r"pairwise $|\Delta Q_i|/\langle Q_i\rangle$"),
                  Line2D([], [], color="0.3", marker="^", lw=0, ms=7, mec="k",
                         label=f"{FLUCT_STAT} (scaling-law anchor)")]
curve_legend = ax_p.legend(loc="upper right")
ax_p.add_artist(curve_legend)
ax_p.legend(handles=marker_handles, loc="lower left")
ax_p.set_ylim(0.0, 115.0 * max(float(np.max([tls_fluct_frac(p, "linear") for p in p_grid])),
                               FLUCT_F1, FLUCT_F2))
ax_p.set_xlabel("Device power (dBm)")
ax_p.set_ylabel(r"$|\Delta Q_i|\,/\,\langle Q_i\rangle$ (%)")
ax_p.set_title("Pairwise quality-factor differences and the fluctuation scaling laws", fontsize=10)
fig.tight_layout()
fig.savefig("timesweep_deltaQi.png", dpi=300)
plt.show()

print("model predictions at the dataset extremes:")
refs = [(float(p_all.min()), "lowest scan"), (FLUCT_P1, "time_sweep_001"),
        (FLUCT_P2, "time_sweep_003"), (float(p_all.max()), "highest scan")]
for p_ref, tag in refs:
    row = "   ".join(f"{m}: {100*tls_fluct_frac(p_ref, m):7.1f}%" for m in ("linear", "step"))
    print(f"  {tag:15s} {p_ref:+7.1f} dBm   {row}")

## Step 10 — Weighted TLS fit

$$\frac{1}{Q_i}(P) = \frac{\delta_{\rm TLS}}{(1+P/P_c)^{\beta}} + \delta_0$$

Fit in $\log_{10}$ space, with each scan weighted by $1/\sigma_{\rm total}^2$
from Steps 8–9 — covariance and TLS fluctuation in quadrature — so noisier,
low-power scans naturally count for less than clean, high-power ones. $\beta$
is left as a free parameter.

A second fit then freezes $\beta$ at $0.5$, the exact prediction of the
non-interacting standard tunneling model. The $\chi^2$ penalty against the
free-$\beta$ fit quantifies how strongly the data prefer their own exponent.


In [ ]:
def tls_log10(th, plin):
    d_tls, pc, beta, d0 = 10 ** th[0], 10 ** (th[1] / 10.0), th[2], 10 ** th[3]
    return np.log10(d_tls / (1.0 + plin / pc) ** beta + d0)

def fit_tls_free(p_dbm, inv_qi, sig_log):
    plin = 10.0 ** (p_dbm / 10.0)
    y = np.log10(inv_qi)
    lo_d, hi_d = np.min(inv_qi), np.max(inv_qi)
    th0 = np.array([np.log10(max(hi_d - lo_d, 0.2 * hi_d)), np.median(p_dbm), 0.3, np.log10(0.8 * lo_d)])
    lb = [np.log10(hi_d) - 6, p_dbm.min() - 30.0, 0.02, np.log10(lo_d) - 4]
    ub = [np.log10(hi_d) + 3, p_dbm.max() + 30.0, 3.00, np.log10(hi_d) + 1]
    def resid(th):
        return (tls_log10(th, plin) - y) / sig_log
    sol = least_squares(resid, np.clip(th0, lb, ub), bounds=(lb, ub), ftol=1e-13, xtol=1e-13, max_nfev=3000)
    dof = max(len(y) - 4, 1)
    chi2red = 2.0 * sol.cost / dof
    try:
        cov = np.linalg.inv(sol.jac.T @ sol.jac)
    except np.linalg.LinAlgError:
        cov = np.full((4, 4), np.nan)
    sbeta = np.sqrt(abs(cov[2, 2]))
    return sol.x, sol.x[2], sbeta, sbeta * np.sqrt(max(chi2red, 1.0)), chi2red

TLS = {}
for res in CONFIG:
    fin = FINAL[res]
    p  = np.array([r["p_dev"]     for r in fin])
    iq = np.array([r["inv_qi"]    for r in fin])
    st = np.array([r["sig_total"] for r in fin])
    th, beta, sbeta, sbeta_s, chi2red = fit_tls_free(p, iq, st)
    TLS[res] = dict(th=th, beta=beta, sbeta=sbeta, sbeta_s=sbeta_s, chi2red=chi2red, p=p, iq=iq, st=st)
    print(f"[{res}]  n={len(fin)}   beta = {beta:.3f} +- {sbeta:.3f} (stat) +- {sbeta_s:.3f} (chi2-scaled)")
    print(f"        delta_TLS={10**th[0]:.3e}  P_c={th[1]:+.1f} dBm  delta_0={10**th[3]:.3e}  chi2/dof={chi2red:.2f}\n")

# --- comparison: beta frozen at the standard-tunneling-model value ---
# The non-interacting standard tunneling model predicts 1/Q_TLS ~ (1 + P/P_c)^(-1/2),
# i.e. beta = 0.5 exactly. Refit with beta fixed there; the chi2 penalty against the
# free-beta fit above quantifies how much the data actually prefer its own exponent.
BETA_FIXED = 0.5

def fit_tls_fixed_beta(p_dbm, inv_qi, sig_log, beta_fixed):
    plin = 10.0 ** (p_dbm / 10.0)
    y = np.log10(inv_qi)
    lo_d, hi_d = np.min(inv_qi), np.max(inv_qi)
    th0 = np.array([np.log10(max(hi_d - lo_d, 0.2 * hi_d)), np.median(p_dbm), np.log10(0.8 * lo_d)])
    lb = [np.log10(hi_d) - 6, p_dbm.min() - 30.0, np.log10(lo_d) - 4]
    ub = [np.log10(hi_d) + 3, p_dbm.max() + 30.0, np.log10(hi_d) + 1]
    def resid(th):
        th4 = np.array([th[0], th[1], beta_fixed, th[2]])
        return (tls_log10(th4, plin) - y) / sig_log
    sol = least_squares(resid, np.clip(th0, lb, ub), bounds=(lb, ub),
                        ftol=1e-13, xtol=1e-13, max_nfev=3000)
    dof = max(len(y) - 3, 1)
    th4 = np.array([sol.x[0], sol.x[1], beta_fixed, sol.x[2]])
    return th4, 2.0 * sol.cost / dof

print(f"--- comparison with beta fixed at {BETA_FIXED} (standard tunneling model) ---")
for res in CONFIG:
    R = TLS[res]
    th_fixed, chi2red_fixed = fit_tls_fixed_beta(R["p"], R["iq"], R["st"], BETA_FIXED)
    chi2_free = R["chi2red"] * max(len(R["iq"]) - 4, 1)
    chi2_fixed = chi2red_fixed * max(len(R["iq"]) - 3, 1)
    R["th_fixed"], R["chi2red_fixed"] = th_fixed, chi2red_fixed
    print(f"[{res}]  delta_TLS={10**th_fixed[0]:.3e}  P_c={th_fixed[1]:+.1f} dBm  "
          f"delta_0={10**th_fixed[3]:.3e}  chi2/dof={chi2red_fixed:.2f}   "
          f"(free beta={R['beta']:.3f}: chi2/dof={R['chi2red']:.2f}, Delta_chi2={chi2_fixed - chi2_free:+.1f})")


## Step 11 — Publication plots

$1/Q_i$ against device power, with the fitted TLS curve overlaid, for each
resonator. Markers encode the measurement tier; error bars are
$\sigma_{\rm total}$ from Steps 8–9.

For completeness the plot shows $1/Q_i$ at **every measured power**: scans
dropped by the Step 7 filters appear as unlabelled grey crosses (clipped to
the frame where they lie far outside the kept range). They are shown only —
they do not enter the fit.


In [ ]:
# Scans grouped by their exact instrument settings (attenuator, IF bandwidth):
# up to five groups across the dataset, styled consistently for both resonators.
GROUP_MARKERS = ("o", "s", "^", "v", "D", "P")
GROUP_COLOURS = ("#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9")

def scan_group(r):
    return (r["atten"], r["bw"])

def group_sort_key(g):
    return (-g[0], -g[1])

def group_label(g):
    return f"{g[0]:.0f} dB atten, {g[1]:.0f} Hz IFBW"

ALL_GROUPS = sorted({scan_group(r) for res in CONFIG for r in FINAL[res]}, key=group_sort_key)
GROUP_STYLE = {}
for k, g in enumerate(ALL_GROUPS):
    GROUP_STYLE[g] = (GROUP_MARKERS[k % len(GROUP_MARKERS)],
                      GROUP_COLOURS[k % len(GROUP_COLOURS)], group_label(g))

for res in CONFIG:
    fin, R = FINAL[res], TLS[res]
    fr_med = np.median([r["fr"] for r in fin])
    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=140)
    for grp, (mk, col, lab) in GROUP_STYLE.items():
        sel = [r for r in fin if scan_group(r) == grp]
        if not sel:
            continue
        x  = np.array([r["p_dev"]     for r in sel])
        yv = np.array([r["inv_qi"]    for r in sel])
        s  = np.array([r["sig_total"] for r in sel])
        yerr = np.array([yv - yv / 10 ** s, yv * 10 ** s - yv])
        ax.errorbar(x, yv, yerr=yerr, fmt="none", ecolor="0.65", elinewidth=0.8, capsize=0, zorder=1)
        ax.plot(x, yv, mk, ms=5, mfc=col, mec="k", mew=0.4, lw=0, label=lab, zorder=3)
    dropped_all = [fit for fit in FITS[res] if not fit["keep"]]
    dropped = [fit for fit in dropped_all if np.isfinite(fit["inv_qi"]) and fit["inv_qi"] > 0]
    kept_lo = min(r["inv_qi"] for r in fin) / 2.5
    kept_hi = max(r["inv_qi"] for r in fin) * 2.5
    if dropped:
        xd = np.array([fit["p_dev"] for fit in dropped])
        yd = np.array([fit["inv_qi"] for fit in dropped])
        ax.plot(xd, np.clip(yd, kept_lo, kept_hi), "x", ms=4.5, color="0.55", lw=0, zorder=1)
    if len(dropped_all) > len(dropped):
        print(f"[{res}] note: {len(dropped_all) - len(dropped)} dropped scan(s) have non-positive "
              f"1/Qi and cannot be shown on the log axis")
    xx = np.linspace(min(r["p_dev"] for r in fin), max(r["p_dev"] for r in fin), 400)
    ax.plot(xx, 10 ** tls_log10(R["th"], 10 ** (xx / 10.0)), "k-", lw=1.8, zorder=2,
            label=rf"$\beta$ = {R['beta']:.3f} $\pm$ {R['sbeta_s']:.3f}")
    ax.set_yscale("log")
    ax.set_ylim(kept_lo, kept_hi)
    ax.set_xlabel("Device power (dBm)")
    ax.set_ylabel(r"Internal loss  $1/Q_i$")
    ax.set_title(rf"{RES_TITLE[res]} internal loss versus device power   ($f_r$ = {fr_med/1e9:.6f} GHz)")
    ax.legend(frameon=False, fontsize=9)
    ax.tick_params(direction="in", which="both", top=True, right=True)
    ax.minorticks_on()
    fig.tight_layout()
    fig.savefig(f"{res}_invQi_vs_power.png", dpi=300)
    plt.show()

### The same fit, plotted against photon number

$$\langle n_{\rm ph}\rangle = \frac{2}{\hbar\,\omega_r^2}\,\frac{Q_L^2}{Q_c}\,P_{\rm in}$$

using each scan's own fitted $f_r$, $Q_L$, $Q_c$, and its device power
(including the fixed fridge attenuation from Step 1). The TLS curve is refit
natively against $\langle n_{\rm ph}\rangle$ rather than just relabeling the
power-axis fit, since $Q_L/Q_c$ varies scan to scan and folds directly into
photon number — so $\beta$ can come out slightly different here than on the
power axis.


In [ ]:
HBAR = 1.054571817e-34  # J*s

def tls_log10_nph(th, n):
    d_tls, nc, beta, d0 = 10 ** th[0], 10 ** th[1], th[2], 10 ** th[3]
    return np.log10(d_tls / (1.0 + n / nc) ** beta + d0)

def fit_tls_nph(n, inv_qi, sig_log):
    y = np.log10(inv_qi)
    lo_d, hi_d = np.min(inv_qi), np.max(inv_qi)
    th0 = np.array([np.log10(max(hi_d - lo_d, 0.2 * hi_d)), np.log10(np.median(n)), 0.3, np.log10(0.8 * lo_d)])
    lb = [np.log10(hi_d) - 6, np.log10(n.min()) - 6, 0.02, np.log10(lo_d) - 4]
    ub = [np.log10(hi_d) + 3, np.log10(n.max()) + 6, 3.00, np.log10(hi_d) + 1]
    def resid(th):
        return (tls_log10_nph(th, n) - y) / sig_log
    sol = least_squares(resid, np.clip(th0, lb, ub), bounds=(lb, ub), ftol=1e-13, xtol=1e-13, max_nfev=5000)
    dof = max(len(y) - 4, 1)
    chi2red = 2.0 * sol.cost / dof
    try:
        cov = np.linalg.inv(sol.jac.T @ sol.jac)
    except np.linalg.LinAlgError:
        cov = np.full((4, 4), np.nan)
    sbeta = np.sqrt(abs(cov[2, 2]))
    return sol.x, sol.x[2], sbeta, sbeta * np.sqrt(max(chi2red, 1.0)), chi2red

TLS_NPH = {}
for res in CONFIG:
    fin = FINAL[res]
    for r in fin:
        p_watts = 10 ** (r["p_dev"] / 10.0) * 1e-3
        omega_r = 2 * np.pi * r["fr"]
        r["nph"] = (2.0 / (HBAR * omega_r ** 2)) * (r["ql"] ** 2 / r["qc"]) * p_watts

    nph = np.array([r["nph"] for r in fin])
    iq  = np.array([r["inv_qi"]    for r in fin])
    st  = np.array([r["sig_total"] for r in fin])
    th, beta, sbeta, sbeta_s, chi2red = fit_tls_nph(nph, iq, st)
    TLS_NPH[res] = dict(th=th, beta=beta, sbeta=sbeta, sbeta_s=sbeta_s, chi2red=chi2red)
    print(f"[{res}]  n={len(fin)}   beta(vs n_ph) = {beta:.3f} +- {sbeta_s:.3f}   "
          f"n_c = {10**th[1]:.3e}   chi2/dof={chi2red:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), dpi=140)
for ax, res in zip(axes, CONFIG):
    fin, R = FINAL[res], TLS_NPH[res]
    fr_med = np.median([r["fr"] for r in fin])
    for grp, (mk, col, lab) in GROUP_STYLE.items():
        sel = [r for r in fin if scan_group(r) == grp]
        if not sel:
            continue
        x  = np.array([r["nph"]       for r in sel])
        yv = np.array([r["inv_qi"]    for r in sel])
        s  = np.array([r["sig_total"] for r in sel])
        yerr = np.array([yv - yv / 10 ** s, yv * 10 ** s - yv])
        ax.errorbar(x, yv, yerr=yerr, fmt="none", ecolor="0.65", elinewidth=0.8, capsize=0, zorder=1)
        ax.plot(x, yv, mk, ms=5, mfc=col, mec="k", mew=0.4, lw=0, label=lab, zorder=3)
    xx = np.logspace(np.log10(min(r["nph"] for r in fin)), np.log10(max(r["nph"] for r in fin)), 400)
    ax.plot(xx, 10 ** tls_log10_nph(R["th"], xx), "k-", lw=1.8, zorder=2,
            label=rf"$\beta$ = {R['beta']:.3f} $\pm$ {R['sbeta_s']:.3f}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Photon number $\langle n_{\rm ph}\rangle$")
    ax.set_ylabel(r"Internal loss  $1/Q_i$")
    ax.set_title(rf"{RES_TITLE[res]} internal loss tangent versus mean photon number   ($f_r$ = {fr_med/1e9:.6f} GHz)")
    ax.legend(frameon=False, fontsize=9)
    ax.tick_params(direction="in", which="both", top=True, right=True)
fig.tight_layout()
fig.savefig("invQi_vs_photon_number.png", dpi=300)
plt.show()

## Step 12 — SNR convergence and photon lifetimes

Two figures per resonator, after the TLS fit.

**Quality factors vs SNR** — split panels in the style of the Probst *et al.*
convergence figure: $Q_i$ alone on top (red), $Q_c$ and $Q_l$ together below
(blue circles / green triangles), against a linear SNR axis. The dashed line
in each panel is the median over the top-SNR quartile — the converged value,
standing in for the Monte-Carlo original's known truth. $Q_c$ is the clean
convergence diagnostic (coupling is power-independent, so its scatter
collapsing onto the dashed line at high SNR is pure fit behaviour); $Q_i$ also
carries real TLS power dependence, since SNR rises with power.

**Photon lifetimes vs device power** — $\tau_c$ (coupling, blue triangles),
$\tau_L$ (loaded, green squares) and $\tau_i$ (internal, red circles) on one
logarithmic axis, with the loaded lifetime printed at the lowest and highest
measured power.

Error bars on $Q_i$ and $\tau_i$ include the fitted TLS-fluctuation term in
quadrature with the covariance, matching every other $Q_i$ error bar in the
notebook; $Q_c$, $Q_l$ and their lifetimes carry pure fit covariance.


In [ ]:
# SNR convergence and photon lifetimes, after the TLS fit.

def q_sigma_total(r):
    """Total Q_i error: covariance (+) the fitted TLS-fluctuation term in
    quadrature, matching every other Q_i error bar in the notebook."""
    return float(np.sqrt(r["sig_qi"] ** 2 + (tls_fluct_frac(r["p_dev"]) * r["qi"]) ** 2))

# --- Q_i (top panel) and Q_c + Q_l (bottom panel) against scan SNR ---
for res in CONFIG:
    fin = FINAL[res]
    snr = np.array([r["snr"] for r in fin])
    top_quartile = snr >= np.quantile(snr, 0.75)
    qi = np.array([r["qi"] for r in fin])
    sig_qi_tot = np.array([q_sigma_total(r) for r in fin])
    qc = np.array([r["qc"] for r in fin])
    sig_qc = np.array([r["sig_qc"] for r in fin])
    ql = np.array([r["ql"] for r in fin])
    sig_ql = np.array([r["sig_ql"] for r in fin])

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(7.2, 6.4), dpi=140, sharex=True,
                                         gridspec_kw=dict(hspace=0.07))
    ax_top.errorbar(snr, qi, yerr=sig_qi_tot, fmt="o", color="#D55E00", ms=3.5, lw=0,
                    elinewidth=0.7, capsize=1.5, ecolor="#D55E00", label=r"$Q_i$")
    ax_top.axhline(np.median(qi[top_quartile]), color="k", ls="--", lw=1.0)
    ax_top.set_ylabel(r"$Q_i$")
    ax_top.legend(fontsize=9, loc="upper right")

    ax_bot.errorbar(snr, qc, yerr=sig_qc, fmt="o", color="#0072B2", ms=3.5, lw=0,
                    elinewidth=0.7, capsize=1.5, ecolor="#0072B2", label=r"$Q_c$")
    ax_bot.errorbar(snr, ql, yerr=sig_ql, fmt="^", color="#009E73", ms=3.5, lw=0,
                    elinewidth=0.7, capsize=1.5, ecolor="#009E73", label=r"$Q_l$")
    ax_bot.axhline(np.median(qc[top_quartile]), color="k", ls="--", lw=1.0)
    ax_bot.axhline(np.median(ql[top_quartile]), color="k", ls="--", lw=1.0)
    ax_bot.set_ylabel(r"$Q_c$ / $Q_l$")
    ax_bot.set_xlabel("SNR (Probst Eq. 14, dimensionless)")
    ax_bot.legend(fontsize=9, loc="upper right")
    for a in (ax_top, ax_bot):
        a.tick_params(direction="in", which="both", top=True, right=True)
        a.minorticks_on()
    fig.suptitle(f"{RES_TITLE[res]} quality factors versus signal-to-noise ratio "
                 "(dashed: top-SNR-quartile median)", fontsize=10)
    fig.savefig(f"{res}_q_vs_snr.png", dpi=300, bbox_inches="tight")
    plt.show()

# --- photon lifetimes vs device power: tau_c, tau_L, tau_i on one axis ---
def get_p_dev(r):
    return r["p_dev"]

def tau_us(r, key):
    return r[key] / (2.0 * np.pi * r["fr"]) * 1e6

for res in CONFIG:
    fin = sorted(FINAL[res], key=get_p_dev)
    p = np.array([r["p_dev"] for r in fin])
    tau_c = np.array([tau_us(r, "qc") for r in fin])
    tau_l = np.array([tau_us(r, "ql") for r in fin])
    tau_i = np.array([tau_us(r, "qi") for r in fin])
    sig_tau_i = np.array([q_sigma_total(r) / (2.0 * np.pi * r["fr"]) * 1e6 for r in fin])

    fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=140)
    ax.plot(p, tau_c, "^", color="#0072B2", ms=4, lw=0, label=r"$\tau_c$ (coupling)")
    ax.plot(p, tau_l, "s", color="#009E73", ms=4, lw=0, label=r"$\tau_L$ (loaded)")
    ax.errorbar(p, tau_i, yerr=sig_tau_i, fmt="o", color="#D55E00", ms=4, lw=0,
                elinewidth=0.7, capsize=1.5, ecolor="#D55E00", label=r"$\tau_i$ (internal)")
    ax.set_yscale("log")
    ax.set_xlabel("Device power (dBm)")
    ax.set_ylabel(r"Photon lifetime $\tau$ ($\mu$s)")
    ax.set_title(f"{RES_TITLE[res]} photon lifetime versus device power")
    ax.legend(fontsize=9)
    ax.tick_params(direction="in", which="both", top=True, right=True)
    fig.tight_layout()
    fig.savefig(f"{res}_lifetime_vs_power.png", dpi=300)
    plt.show()
    print(f"[{res}] Low Power:  tau_L = {tau_l[0]:.0f} us   (P_dev = {p[0]:+.1f} dBm)")
    print(f"[{res}] High Power: tau_L = {tau_l[-1]:.0f} us   (P_dev = {p[-1]:+.1f} dBm)")

## Step 13 — Summary

$\beta$, fit quality, and the calibrated cable delays for both resonators,
side by side.


In [ ]:
print(f"{'':6s} {'n':>4s} {'beta':>16s} {'chi2/dof':>10s} {'tau(-30dB)':>11s} {'tau(-70dB)':>11s}")
for res in CONFIG:
    R, t30, t70 = TLS[res], *TAUS[res]
    print(f"{res:6s} {len(FINAL[res]):4d} {R['beta']:8.3f} +- {R['sbeta_s']:.3f} "
          f"{R['chi2red']:10.2f} {t30*1e9:+10.2f}n {t70*1e9:+10.2f}n")
print(f"\ntotal runtime since fitting started: {time.time() - t_start:.1f} s")

## Step 14 — Individual fits: magnitude, phase and IQ, one plot per scan

Every scan, shown individually: magnitude and phase stacked on the left
(frequency referenced to that scan's own fitted $f_r$), and the IQ plane on
the right with the fitted circle overlaid. Grey dots are the calibrated data;
the blue curve is the model. $Q_i$, $Q_L$, and $Q_c$ are annotated with their
own per-fit covariance errors (the ensemble TLS-fluctuation term of Step 9
describes drift between repeated measurements, so it doesn't apply to a
single-scan fit readout), along with SNR, and the title reports whether the
scan was **KEPT** or **DROPPED** (and why, from Step 7).

Data points are decimated for display only, not for fitting, to keep the
notebook a reasonable size.


In [ ]:
def decim(x, max_pts=300):
    stride = max(1, len(x) // max_pts)
    return x[::stride]

def get_device_power(fit_record):
    return fit_record["p_dev"]

n_shown = 0
for res in CONFIG:
    for s in DATA[res]:
        scans_for_this_sweep = [r for r in FITS[res] if r["sweep"] == s["name"]]
        rows = sorted(scans_for_this_sweep, key=get_device_power)
        print(f"----- {res} : {s['name']} ({len(rows)} scans) -----")
        for r in rows:
            f, zn, model = r["f"], r["zn"], r["zn_model"]
            fr_fit = r["fr"]
            idx = decim(np.arange(len(f)))
            df_khz = (f[idx] - fr_fit) / 1e3

            fig, axd = plt.subplot_mosaic([["mag", "iq"], ["phase", "iq"]], figsize=(9.2, 4.3),
                                          dpi=100, gridspec_kw=dict(width_ratios=[1, 1.15],
                                                                    hspace=0.35, wspace=0.32))
            ax_mag, ax_ph, ax_iq = axd["mag"], axd["phase"], axd["iq"]

            ax_mag.plot(df_khz, 20 * np.log10(np.abs(zn[idx]) + 1e-30), ".", ms=2.5, color="0.4", alpha=0.6)
            ax_mag.plot(df_khz, 20 * np.log10(np.abs(model[idx]) + 1e-30), "-", color="#0072B2", lw=1.6)
            ax_mag.set_ylabel(r"$|S_{21}|$ (dB)")
            ax_mag.set_title(f"{r['p_dev']:+.1f} dBm [{r['tier']}]", fontsize=9)

            ax_ph.plot(df_khz, np.degrees(np.unwrap(np.angle(zn[idx]))), ".", ms=2.5, color="0.4", alpha=0.6)
            ax_ph.plot(df_khz, np.degrees(np.unwrap(np.angle(model[idx]))), "-", color="#0072B2", lw=1.6)
            ax_ph.set_xlabel(r"$f - f_r$ (kHz)")
            ax_ph.set_ylabel(r"$\arg S_{21}$ (deg)")

            ax_iq.plot(zn[idx].real, zn[idx].imag, ".", ms=3, color="0.4", alpha=0.5)
            ax_iq.plot(model[idx].real, model[idx].imag, "-", color="#0072B2", lw=1.8)
            ax_iq.set_aspect("equal", adjustable="datalim")
            ax_iq.set_xlabel(r"Re $S_{21}$ (normalised)")
            ax_iq.set_ylabel(r"Im $S_{21}$ (normalised)")
            rel = 100 * r['sig_qi'] / r['qi']
            rel_ql = 100 * r['sig_ql'] / r['ql']
            rel_qc = 100 * r['sig_qc'] / r['qc']
            ax_iq.text(0.97, 0.95,
                       f"Qi={r['qi']:.2g} \u00b1 {r['sig_qi']:.1g} ({rel:.0f}%)\n"
                       f"QL={r['ql']:.2g} \u00b1 {r['sig_ql']:.1g} ({rel_ql:.0f}%)\n"
                       f"Qc={r['qc']:.2g} \u00b1 {r['sig_qc']:.1g} ({rel_qc:.0f}%)\n"
                       f"SNR={r['snr']:.2f}",
                       transform=ax_iq.transAxes,
                       ha="right", va="top", fontsize=8.5,
                       bbox=dict(boxstyle="round", fc="white", ec="0.6", alpha=0.85))

            status = "KEPT" if r["keep"] else f"DROPPED: {r['why']}"
            fig.suptitle(f"{RES_TITLE[res]} — single-scan fit at {r['p_dev']:+.1f} dBm device power   [{status}]", fontsize=10)
            fig.tight_layout(rect=[0, 0, 1, 0.90])
            plt.show()
            plt.close(fig)
            n_shown += 1
print(f"\ntotal individual fit plots shown: {n_shown}")